<a href="https://colab.research.google.com/github/codingNR29/Gesture-controlled-robot-arm/blob/main/Gesture_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CNN-BASED GESTURE CLASSIFICATION MODEL FOR IMU DATA
# ============================================================
# This code is written to be easy to understand.
# Read the comments slowly from top to bottom.
#
# ASSUMPTION:
# Each gesture sample contains 450 numbers total.
# We assume those 450 numbers are:
#     50 time steps  x  9 sensor features
#
# Example 9 features at each time step:
#     ax, ay, az, gx, gy, gz, mx, my, mz
#
# So instead of feeding the model a flat shape like (450,),
# we reshape the data into (50, 9).
#
# WHY?
# Because a CNN works better when it can "see" the time structure
# of the gesture rather than one long flat list.
# ============================================================

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# ------------------------------------------------------------
# 1. BASIC SETTINGS
# ------------------------------------------------------------

# Number of time steps in one gesture sample
TIMESTEPS = 50

# Number of features per time step
# For MPU9250 this is often:
# 3 accel + 3 gyro + 3 magnetometer = 9
FEATURES = 9

# Number of gesture classes
# Example:
# 0 = Idle
# 1 = Wave
# 2 = Punch
# 3 = Grab
NUM_CLASSES = 4

# Training settings
EPOCHS = 100
BATCH_SIZE = 32

# ------------------------------------------------------------
# 2. LOAD OR PREPARE YOUR DATA
# ------------------------------------------------------------
# You must replace these placeholder lines with your real data.
#
# Your X data can be in ONE of these forms:
#
# OPTION A:
# X_train shape = (number_of_samples, 450)
# X_val   shape = (number_of_samples, 450)
# X_test  shape = (number_of_samples, 450)
#
# OPTION B:
# X_train shape = (number_of_samples, 50, 9)
# X_val   shape = (number_of_samples, 50, 9)
# X_test  shape = (number_of_samples, 50, 9)
#
# y arrays should contain integer labels like:
# y_train = [0, 1, 3, 2, 0, ...]
#
# ------------------------------------------------------------
# EXAMPLE PLACEHOLDERS:
# Uncomment and replace with your own loading code.
# ------------------------------------------------------------

# X_train = np.load("X_train.npy")
# y_train = np.load("y_train.npy")
# X_val   = np.load("X_val.npy")
# y_val   = np.load("y_val.npy")
# X_test  = np.load("X_test.npy")
# y_test  = np.load("y_test.npy")

# ------------------------------------------------------------
# If you instead have ALL data in one file and want to split it,
# you can do something like this:
# ------------------------------------------------------------

# X = np.load("X.npy")
# y = np.load("y.npy")

# ------------------------------------------------------------
# 3. HELPER FUNCTION TO RESHAPE INPUT DATA
# ------------------------------------------------------------
# This function makes sure your data is in the correct CNN shape.
# If your data is flat (450,), it reshapes to (50, 9).
# If your data is already (50, 9), it keeps it as it is.
# ------------------------------------------------------------

def prepare_input_data(X, timesteps=TIMESTEPS, features=FEATURES):
    """
    Convert input data to the shape expected by Conv1D:
        (number_of_samples, timesteps, features)

    Accepted input shapes:
        (N, 450)     -> reshaped to (N, 50, 9)
        (N, 50, 9)   -> kept as is
    """
    X = np.asarray(X, dtype=np.float32)

    # Case 1: data is flat, for example (N, 450)
    if X.ndim == 2:
        expected_total_features = timesteps * features

        if X.shape[1] != expected_total_features:
            raise ValueError(
                f"Expected each sample to have {expected_total_features} values, "
                f"but got {X.shape[1]}. "
                f"Check TIMESTEPS and FEATURES."
            )

        # Reshape from (N, 450) to (N, 50, 9)
        X = X.reshape(-1, timesteps, features)

    # Case 2: data is already in time-series form, for example (N, 50, 9)
    elif X.ndim == 3:
        if X.shape[1] != timesteps or X.shape[2] != features:
            raise ValueError(
                f"Expected shape (N, {timesteps}, {features}), "
                f"but got {X.shape}."
            )

    # Any other shape is invalid for this model
    else:
        raise ValueError(
            f"Input must have 2 or 3 dimensions, but got shape {X.shape}."
        )

    return X

# ------------------------------------------------------------
# 4. OPTIONAL HELPER FUNCTION TO SPLIT DATA
# ------------------------------------------------------------
# Use this ONLY if you have one big dataset X and y and want
# to split it into train / validation / test.
#
# If you already have X_train, X_val, X_test separately,
# you do not need this function.
# ------------------------------------------------------------

def split_data(X, y, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, seed=42):
    """
    Split data into train / validation / test sets.
    """
    # Safety check to make sure ratios add up to 1
    if not np.isclose(train_ratio + val_ratio + test_ratio, 1.0):
        raise ValueError("train_ratio + val_ratio + test_ratio must equal 1.0")

    # Convert to numpy arrays
    X = np.asarray(X)
    y = np.asarray(y)

    # Shuffle indices so the split is random
    rng = np.random.default_rng(seed)
    indices = np.arange(len(X))
    rng.shuffle(indices)

    X = X[indices]
    y = y[indices]

    # Compute split points
    train_end = int(len(X) * train_ratio)
    val_end = train_end + int(len(X) * val_ratio)

    # Slice the arrays
    X_train = X[:train_end]
    y_train = y[:train_end]

    X_val = X[train_end:val_end]
    y_val = y[train_end:val_end]

    X_test = X[val_end:]
    y_test = y[val_end:]

    return X_train, y_train, X_val, y_val, X_test, y_test

# ------------------------------------------------------------
# 5. BUILD THE CNN MODEL
# ------------------------------------------------------------
# This is the heart of the code.
#
# HOW THE MODEL WORKS:
#
# Input (50, 9)
#     ->
# Conv1D layer learns small motion patterns over time
#     ->
# MaxPooling reduces size and keeps strong features
#     ->
# Another Conv1D learns more meaningful patterns
#     ->
# GlobalAveragePooling compresses the time dimension
#     ->
# Dense layer makes final decision
#     ->
# Softmax gives probabilities for 4 gesture classes
# ------------------------------------------------------------

def build_gesture_cnn(normalizer):
    """
    Build a 1D CNN for gesture classification.
    """

    model = models.Sequential([
        # ----------------------------------------------------
        # Input layer
        # Each sample is a sequence of 50 time steps
        # and each time step has 9 features
        # ----------------------------------------------------
        layers.Input(shape=(TIMESTEPS, FEATURES)),

        # ----------------------------------------------------
        # Normalization layer
        # This helps the model train better by standardizing
        # the input features.
        #
        # We will "adapt" this layer on the training data later.
        # ----------------------------------------------------
        normalizer,

        # ----------------------------------------------------
        # First convolution layer
        # 16 filters means 16 small pattern detectors
        # kernel_size=3 means each detector looks at 3 time steps
        # at a time
        # padding='same' keeps the output length similar
        # ----------------------------------------------------
        layers.Conv1D(
            filters=16,
            kernel_size=3,
            activation='relu',
            padding='same'
        ),

        # ----------------------------------------------------
        # Pooling layer
        # This reduces the time dimension by half
        # and keeps the strongest features
        # ----------------------------------------------------
        layers.MaxPooling1D(pool_size=2),

        # ----------------------------------------------------
        # Second convolution layer
        # More filters = model can learn more patterns
        # ----------------------------------------------------
        layers.Conv1D(
            filters=32,
            kernel_size=3,
            activation='relu',
            padding='same'
        ),

        # ----------------------------------------------------
        # Another pooling layer
        # ----------------------------------------------------
        layers.MaxPooling1D(pool_size=2),

        # ----------------------------------------------------
        # Third convolution layer
        # This helps the model learn slightly higher-level
        # motion features
        # ----------------------------------------------------
        layers.Conv1D(
            filters=32,
            kernel_size=3,
            activation='relu',
            padding='same'
        ),

        # ----------------------------------------------------
        # GlobalAveragePooling1D
        # Instead of flattening a large tensor, this takes the
        # average of each feature map over time.
        #
        # It reduces parameters and is usually good for TinyML.
        # ----------------------------------------------------
        layers.GlobalAveragePooling1D(),

        # ----------------------------------------------------
        # Dense layer
        # Combines the features extracted by the CNN
        # ----------------------------------------------------
        layers.Dense(32, activation='relu'),

        # ----------------------------------------------------
        # Dropout
        # Randomly turns off some neurons during training
        # to reduce overfitting
        # ----------------------------------------------------
        layers.Dropout(0.2),

        # ----------------------------------------------------
        # Output layer
        # 4 neurons for 4 classes
        # Softmax turns outputs into probabilities
        # ----------------------------------------------------
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    return model

# ------------------------------------------------------------
# 6. MAIN TRAINING LOGIC
# ------------------------------------------------------------
# You have two ways to use this section:
#
# WAY A:
# If you already have:
#   X_train, y_train, X_val, y_val, X_test, y_test
# then just prepare them and continue.
#
# WAY B:
# If you have only X and y,
# use split_data(X, y) first.
# ------------------------------------------------------------

if __name__ == "__main__":

    # ========================================================
    # IMPORTANT:
    # Replace this example block with your real data loading.
    # ========================================================

    # --------------------------------------------------------
    # EXAMPLE DEMO DATA
    # --------------------------------------------------------
    # This is ONLY so the script is complete and runnable.
    # DELETE this block when you use your real dataset.
    # --------------------------------------------------------
    num_samples = 500

    # Fake random data just for demonstration
    # Shape here is (500, 450), like your old dense model input
    X = np.random.randn(num_samples, TIMESTEPS * FEATURES).astype(np.float32)

    # Fake labels from 0 to 3
    y = np.random.randint(0, NUM_CLASSES, size=(num_samples,))

    # Split fake data
    X_train, y_train, X_val, y_val, X_test, y_test = split_data(X, y)

    # --------------------------------------------------------
    # PREPARE THE INPUT SHAPES
    # --------------------------------------------------------
    # If data is (N, 450), this will convert it to (N, 50, 9)
    # --------------------------------------------------------
    X_train = prepare_input_data(X_train)
    X_val   = prepare_input_data(X_val)
    X_test  = prepare_input_data(X_test)

    # Make sure labels are integer type
    y_train = np.asarray(y_train, dtype=np.int32)
    y_val   = np.asarray(y_val, dtype=np.int32)
    y_test  = np.asarray(y_test, dtype=np.int32)

    # --------------------------------------------------------
    # CREATE THE NORMALIZATION LAYER
    # --------------------------------------------------------
    # This layer learns the mean and variance from X_train only.
    # That is the correct thing to do.
    # --------------------------------------------------------
    normalizer = layers.Normalization(axis=-1)

    # "adapt" means: learn statistics from training data
    normalizer.adapt(X_train)

    # --------------------------------------------------------
    # BUILD THE MODEL
    # --------------------------------------------------------
    my_model = build_gesture_cnn(normalizer)

    # --------------------------------------------------------
    # COMPILE THE MODEL
    # --------------------------------------------------------
    # sparse_categorical_crossentropy is correct because:
    # labels are integers like 0, 1, 2, 3
    #
    # Adam is a very common optimizer and works well here.
    # --------------------------------------------------------
    my_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # Print the model architecture
    my_model.summary()

    # --------------------------------------------------------
    # CALLBACKS
    # --------------------------------------------------------
    # EarlyStopping:
    # Stops training if validation loss does not improve.
    #
    # ReduceLROnPlateau:
    # Reduces learning rate if validation loss gets stuck.
    # --------------------------------------------------------
    early_stopping = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )

    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-5
    )

    # --------------------------------------------------------
    # TRAIN THE MODEL
    # --------------------------------------------------------
    history = my_model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )

    # --------------------------------------------------------
    # EVALUATE THE MODEL ON TEST DATA
    # --------------------------------------------------------
    test_loss, test_accuracy = my_model.evaluate(X_test, y_test, verbose=0)

    print("\nTest Loss:", test_loss)
    print("Test Accuracy:", test_accuracy)

    # --------------------------------------------------------
    # MAKE PREDICTIONS
    # --------------------------------------------------------
    # This gives probabilities for each class.
    # Example output for one sample:
    # [0.02, 0.91, 0.03, 0.04]
    # meaning class 1 has the highest probability
    # --------------------------------------------------------
    predictions = my_model.predict(X_test[:5])

    print("\nPredictions for first 5 test samples:")
    print(predictions)

    # Most likely class index for each prediction
    predicted_classes = np.argmax(predictions, axis=1)

    print("\nPredicted classes:")
    print(predicted_classes)

    print("\nTrue classes:")
    print(y_test[:5])

    # --------------------------------------------------------
    # SAVE THE TRAINED KERAS MODEL
    # --------------------------------------------------------
    # This saves the normal Keras model
    # --------------------------------------------------------
    my_model.save("gesture_cnn_model.keras")

    # --------------------------------------------------------
    # OPTIONAL: CONVERT TO TFLITE
    # --------------------------------------------------------
    # This is useful later if you want deployment on embedded
    # devices. This basic conversion is a good start.
    # --------------------------------------------------------
    converter = tf.lite.TFLiteConverter.from_keras_model(my_model)

    # Basic optimization to reduce model size
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    tflite_model = converter.convert()

    with open("gesture_cnn_model.tflite", "wb") as f:
        f.write(tflite_model)

    print("\nTFLite model saved as gesture_cnn_model.tflite")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization (Normalization)   │ (None, 50, 9)          │            19 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 50, 16)         │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 25, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 25, 32)         │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 12, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 12, 32)         │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,327 (24.72 KB)

 Trainable params: 6,308 (24.64 KB)

 Non-trainable params: 19 (80.00 B)

Epoch 1/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.2200 - loss: 1.4260 - val_accuracy: 0.2000 - val_loss: 1.3956 - learning_rate: 0.0010
Epoch 2/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2571 - loss: 1.3846 - val_accuracy: 0.2800 - val_loss: 1.3820 - learning_rate: 0.0010
Epoch 3/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2314 - loss: 1.3940 - val_accuracy: 0.2400 - val_loss: 1.3965 - learning_rate: 0.0010
Epoch 4/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2514 - loss: 1.3923 - val_accuracy: 0.2400 - val_loss: 1.3891 - learning_rate: 0.0010
Epoch 5/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2657 - loss: 1.3810 - val_accuracy: 0.2500 - val_loss: 1.3912 - learning_rate: 0.0010
Epoch 6/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3057 - loss: 1.3782 - val_accuracy: 0.2500 - val_loss: 1.3873 - learning_rate: 0.0010
Epoch 7/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2857 - loss: 1.3764 - val